<a href="https://colab.research.google.com/github/NicolasHillingdale/Part1-Basic_Agent_Part_1_Colab.ipynb/blob/test/Part1/Basic_Agent_Part_1_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Last Updated: August 2026
(Tested on Google Colab)

In [20]:
!python --version

Python 3.12.13


In [3]:
!pip install -q -U "tavily-python>=0.7.0" "crewai[tools]" "python-dotenv" "langchain-anthropic" "pandas" "langchain_community"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.2/197.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 836.4/836.4 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.5/60.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.

In [ ]:
# Restart code
import os
os.kill(os.getpid(), 9)

In [21]:
from google.colab import userdata
import os
import crewai
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from tavily import TavilyClient
import json

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [22]:
llm = LLM(model="anthropic/claude-sonnet-4-5-20250929")
client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

MAX_SNIPPET = 400   # characters of context per result

@tool("Tavily Search")
def tavily_search(query: str) -> str:
    """Search recent news for renewable energy deals. Returns compact JSON: title, url, date, snippet."""
    r = client.search(
        query=query,
        search_depth="basic",        # 'advanced' returns much more text
        topic="news",
        time_range="week",
        max_results=5,               # was 10
        include_raw_content=False,   # <-- the main fix
        include_answer=False,
        exclude_domains=["linkedin.com", "facebook.com", "youtube.com"],
    )
    lean = [
        {
            "title": x.get("title", "")[:150],
            "url": x.get("url", ""),
            "date": x.get("published_date"),
            "snippet": (x.get("content") or "")[:MAX_SNIPPET],
        }
        for x in r.get("results", [])
    ]
    return json.dumps(lean, ensure_ascii=False)

@tool("Tavily Extract")
def tavily_extract(url: str) -> str:
    """Fetch the main text of ONE article URL. Returns at most 3000 characters."""
    res = client.extract([url.strip()])
    chunks = [c.get("raw_content", "") for c in res.get("results", [])]
    return ("\n\n".join(chunks))[:3000]

In [23]:
from pydantic import BaseModel, Field
from typing import List, Optional

class Opportunity(BaseModel):
  company_name: str
  country: str
  region: str
  technology: str
  deal_type: str
  stage: str
  signal: str
  deal_size: Optional[str] = None
  source_url: str
  date_announced: Optional[str] = None

class OpportunityList(BaseModel):
  opportunities: List[Opportunity]

class ScoredOpportunity(Opportunity):
  mandate_fit_score: int = Field(ge=1, le=10)
  financing_need_score: int = Field(ge=1, le=10)
  engagement_likelihood_score: int = Field(ge=1, le=10)
  overall_score: float
  rationale: str
  suggested_role: str              # Advisor / Equity / Debt / Monitor

class ScoredList(BaseModel):
  scored: List[ScoredOpportunity]

In [24]:
researcher = Agent(
    role="Renewable Energy Origination Researcher",
    goal=(
        "Find companies, IPPs and project developers in the African and European "
        "renewable energy markets that show evidence of needing project or growth finance."
    ),
    backstory=(
        "You are a research analyst at Hillingdale Capital, an independent capital-raising "
        "and corporate finance advisory firm with offices in Cape Town and Paris. You track "
        "M&A of developers/IPPs/portfolios, project financing and financial close announcements, "
        "equity raises, new utility-scale project announcements, asset sales, and fund closes "
        "targeting African or European renewables — across solar, wind, storage, hydro, hybrid "
        "and green hydrogen, at any stage from development to operational refinancing. "
        "Focus countries: South Africa, Kenya, Nigeria, Egypt, Morocco, Ghana, Senegal, "
        "UK, France, Germany, Spain, Italy. You favour reputable trade and financial press "
        "and you never report a company without a verifiable source URL."
    ),
    tools=[tavily_search, tavily_extract],
    llm=llm,
    allow_delegation=False,
    max_iter=10,
    respect_context_window=True,    # summarises instead of dying at the limit
    verbose=True
)

In [25]:
evaluator = Agent(
    role="Origination Evaluator",
    goal=(
        "Score each researched opportunity against the Hillingdale mandate and judge how "
        "likely it is to need financing and to engage with an independent advisor."
    ),
    backstory=(
        "You are an evaluation analyst at Hillingdale Capital. You are deliberately sceptical: "
        "you discount vague announcements, already-funded projects, and companies large enough "
        "to have in-house capital markets teams or existing tier-1 advisors. You score on three "
        "axes, 1-10 each:\n"
        "  mandate_fit — does this sit inside Hillingdale's sector, geography and instrument set?\n"
        "  financing_need — how strong is the evidence they need capital now (1 = fully funded, "
        "10 = explicitly in-market seeking debt or equity)?\n"
        "  engagement_likelihood — how plausible is it they would work with an independent "
        "advisor (1 = mega-cap with a house bank, 10 = mid-size developer with no named advisor)?\n"
        "overall_score is the mean of the three, rounded to one decimal."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

In [26]:
research = Task(
    description=(
        "Run AT MOST 8 searches, then stop and report. Cover: M&A, project finance/financial "
        "close, equity raises, new utility-scale projects, and fund closes across Africa and "
        "Europe. Use the extract tool only for articles you intend to include — at most 5 times. "
        "Return only companies with a real, verifiable source URL; never invent a company, a "
        "figure or a link. Aim for 10-15 solid entries."
    ),
    expected_output="A structured list of opportunities matching the OpportunityList schema.",
    output_pydantic=OpportunityList,
    agent=researcher,
)

evaluate = Task(
    description=(
        "Score every opportunity from the research task using the three-axis rubric in your "
        "backstory. Give a one-to-two sentence rationale per company and assign a suggested "
        "role for Hillingdale: Advisor, Equity, Debt, or Monitor. Do not add companies that "
        "were not in the research output."
    ),
    expected_output="A structured list matching the ScoredList schema.",
    output_pydantic=ScoredList,
    context=[research],          # <-- this is the handoff, not {companies_list}
    agent=evaluator,
)

In [27]:
crewman = Crew(
    agents=[researcher, evaluator],
    tasks=[research, evaluate],
    process=Process.sequential,
    verbose=True,
)

result = await crewman.kickoff_async()     # <-- the fix

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5b47bfd6-9d41-4b13-9e68-2b91d6f6c05b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Run AT MOST 8 searches, then stop and report. Cover: M&A, project finance/financial close, equity        │
│  raises, new utility-scale projects, and fund closes across Africa and Europe. Use the extract tool only for    │
│  articles you intend to include — at most 5 times. Return only companies with a real, verifiable source URL;    │
│  never invent a company, a figure or a link. Aim for 10-15 solid entries.                                       │
│  ID: c9e682d5-76a1-47c1-a38c-38b31fdb5389                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Renewable Energy Origination Researcher                                                                 │
│                                                                                                                 │
│  Task: Run AT MOST 8 searches, then stop and report. Cover: M&A, project finance/financial close, equity        │
│  raises, new utility-scale projects, and fund closes across Africa and Europe. Use the extract tool only for    │
│  articles you intend to include — at most 5 times. Return only companies with a real, verifiable source URL;    │
│  never invent a company, a figure or a link. Aim for 10-15 solid entries.                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: tavily_search                                                                                            │
│  Args: {'query': 'Africa renewable energy project finance financial close 2024'}                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: tavily_search                                                                                            │
│  Args: {'query': 'Europe solar wind M&A acquisition 2024'}                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Origination Evaluator                                                                                   │
│                                                                                                                 │
│  Task: Score every opportunity from the research task using the three-axis rubric in your backstory. Give a     │
│  one-to-two sentence rationale per company and assign a suggested role for Hillingdale: Advisor, Equity, Debt,  │
│  or Monitor. Do not add companies that were not in the research output.                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Origination Evaluator                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  scored=[ScoredOpportunity(company_name='Scatec', country='Egypt', region='Africa', technology='Solar and       │
│  Battery Storage', deal_type='Project Finance', stage='Pre-Financial Close', signal='Signed 25-year PPA with    │
│  Egyptian Electricity Transmission Company in January 2026, awaiting financial close in H2 2026',               │
│  deal_size='1.95 GW solar + 3.9 GWh batteries (capex undisclosed, to be revealed at financial close)',          │
│  source_url='https://energy-news-network.com/industry-news/scatec-signs-1-95gw-solar-and-3-9gwh-storage-ppa-wi  │
│  th-egypts-transmission-company', date_announced='2026-01-11', mandate_fit_score=9, financing_need_score=8,     │
│  engagement_likelihood_score=4, overall_score=7.0, rationale='Perfect mandate fit (Africa solar/storage,        │
│  utility-scale PPA-backed project finance) with clear H2 2026 financial close timeline. However, Scatec is a    │
│  publicly-traded Norwegian IPP with established banking relationships and in-house capital markets capability,  │
│  reducing engagement likelihood.', suggested_role='Monitor'), ScoredOpportunity(company_name='ARM-Harith        │
│  Infrastructure Equity Fund', country='Nigeria', region='Africa', technology='Multi-technology (electricity     │
│  infrastructure and energy-efficient technologies)', deal_type='Equity Raise', stage='Approved',                │
│  signal='African Development Bank approved $10 million concessional equity investment for West Africa region    │
│  infrastructure', deal_size='$10 million',                                                                      │
│  source_url='https://energy-news-network.com/industry-news/scatec-signs-1-95gw-solar-and-3-9gwh-storage-ppa-wi  │
│  th-egypts-transmission-company', date_announced='2024-12-16', mandate_fit_score=6, financing_need_score=2,     │
│  engagement_likelihood_score=2, overall_score=3.3, rationale='This is a DFI-backed infrastructure fund that     │
│  has already secured its capital from AfDB; no evidence of ongoing fundraising or need for independent          │
│  advisory. The opportunity is already closed.', suggested_role='Monitor'),                                      │
│  ScoredOpportunity(company_name='Africa50 / International Solar Alliance', country='Multi-country Africa',      │
│  region='Africa', technology='Distributed Solar', deal_type='Fund Close / Facility Launch', stage='Term Sheet   │
│  Signed', signal='Africa50 signed term sheet in December 2024 to manage and implement $200 million Africa       │
│  Solar Facility for distributed solar projects', deal_size='$200 million',                                      │
│  source_url='https://thebesanamail.com/2026/08/26/africas-solar-ambitions-get-new-financing-push-as-world-bank  │
│  -backs-distributed-energy', date_announced='2024-12-01', mandate_fit_score=7, financing_need_score=3,          │
│  engagement_likelihood_score=1, overall_score=3.7, rationale='Multi-lateral fund facility managed by Africa50   │
│  (Pan-African infrastructure investor backed by AfDB and sovereign nations) with World Bank involvement. This   │
│  is institutional infrastructure, not an advisory opportunity for an independent firm.',                        │
│  suggested_role='Monitor'), ScoredOpportunity(company_name='Swedfund portfolio companies',                      │
│  country='Multi-country Africa', region='Africa', techn

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Score every opportunity from the research task using the three-axis rubric in your backstory. Give a     │
│  one-to-two sentence rationale per company and assign a suggested role for Hillingdale: Advisor, Equity, Debt,  │
│  or Monitor. Do not add companies that were not in the research output.                                         │
│  Agent: Origination Evaluator                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [30]:
import pandas as pd
from google.colab import files


df = pd.DataFrame([s.model_dump() for s in result.pydantic.scored])
df = df.sort_values("overall_score", ascending=False)
df.to_csv("hillingdale_opportunities.csv", index=False)
files.download("hillingdale_opportunities.csv")

df.head(20)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,company_name,country,region,technology,deal_type,stage,signal,deal_size,source_url,date_announced,mandate_fit_score,financing_need_score,engagement_likelihood_score,overall_score,rationale,suggested_role
0,Scatec,Egypt,Africa,Solar and Battery Storage,Project Finance,Pre-Financial Close,Signed 25-year PPA with Egyptian Electricity T...,1.95 GW solar + 3.9 GWh batteries (capex undis...,https://energy-news-network.com/industry-news/...,2026-01-11,9,8,4,7.0,"Perfect mandate fit (Africa solar/storage, uti...",Monitor
7,UK Offshore Wind Developers (Allocation Round 7),United Kingdom,Europe,Offshore Wind,Government Procurement / CfD Auction,Results Announced,UK awarded 8.4 GW of offshore wind capacity in...,8.4 GW,https://voiceofrenewables.com/cfd-auction-refo...,2026-01-01,9,7,3,6.3,Excellent mandate fit for large-scale European...,Monitor
9,Solaria Energía y Medio Ambiente,Spain,Europe,Solar PV,Development Pipeline,Development / Operating Portfolio,"Pure-play solar PV producer with 14,200 MW pip...","14,200 MW pipeline",https://simplywall.st/stocks/es/utilities/bme-...,2026-08-29,8,6,4,6.0,Strong European solar pipeline with multi-coun...,Advisor
12,MENA Green Hydrogen Developers (China-built pr...,"Multi-country MENA (includes Morocco, Egypt)",Africa / Middle East,Green Hydrogen,Project Development / International Partnership,Development,China building green hydrogen export profile i...,Not disclosed,https://dialogue.earth/en/energy/green-hydroge...,2026-08-24,6,5,2,4.3,Green hydrogen in Africa/MENA fits thematic ma...,Monitor
10,Obudu Capital portfolio companies,Nigeria,Africa,Climate Tech / Renewable Energy,Climate Finance / Venture Capital,Active Investment,Obudu Capital focuses on climate finance in Ni...,Not disclosed,https://creators.spotify.com/pod/profile/the-e...,2026-08-25,6,4,3,4.3,Obudu Capital is itself a climate-focused VC f...,Monitor
5,Fortuna Mining / Africa Power Services,Senegal,Africa,Hybrid (captive power with future renewables i...,Project Development,EPC Contract Awarded,14 MW captive power plant at Diamba Sud Gold P...,14 MW power plant (capex undisclosed),https://cyprusshippingnews.com/2026/08/31/wart...,2026-08-31,4,3,5,4.0,Captive mining power with vague future renewab...,Monitor
2,Africa50 / International Solar Alliance,Multi-country Africa,Africa,Distributed Solar,Fund Close / Facility Launch,Term Sheet Signed,Africa50 signed term sheet in December 2024 to...,$200 million,https://thebesanamail.com/2026/08/26/africas-s...,2024-12-01,7,3,1,3.7,Multi-lateral fund facility managed by Africa5...,Monitor
4,Multiple South African IPPs,South Africa,Africa,"Multi-technology (Solar, Wind, Storage)",Project Finance,Financial Close (2024) / Commercial Operation ...,"28 IPP projects totaling 2,202 MW expected to ...","2,202 MW (28 projects)",https://www.esi-africa.com/business-and-market...,2024-01-01,8,1,2,3.7,Strong mandate fit for South African renewable...,Monitor
1,ARM-Harith Infrastructure Equity Fund,Nigeria,Africa,Multi-technology (electricity infrastructure a...,Equity Raise,Approved,African Development Bank approved $10 million ...,$10 million,https://energy-news-network.com/industry-news/...,2024-12-16,6,2,2,3.3,This is a DFI-backed infrastructure fund that ...,Monitor
11,InnoEnergy portfolio companies,Multi-country Europe,Europe,Multi-technology Clean Tech,Venture Capital / Growth Equity,Active Portfolio,InnoEnergy supports 160+ active portfolio comp...,€35+ billion raised by portfolio (cumulative t...,https://innoenergy.com,2025-12-31,5,3,2,3.3,"InnoEnergy is a VC fund/accelerator, not an op...",Monitor
